In [1]:
from pathlib import Path
import os
import sys
import math
import time
import json
import subprocess
import threading
import statistics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

ROOT = Path("/app")
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"

GPU_INFO_DIR = RESULTS / "gpu_info"
MATMUL_DIR = RESULTS / "matmul"
BANDWIDTH_DIR = RESULTS / "bandwidth"
ATTN_DIR = RESULTS / "attention"
THERMAL_DIR = RESULTS / "thermal"

for p in [GPU_INFO_DIR, MATMUL_DIR, BANDWIDTH_DIR, ATTN_DIR, THERMAL_DIR, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

ModuleNotFoundError: No module named 'pandas'

# HW2.5 — GPU Assignment I: Precision, Bandwidth, and the Cost of Attention

**GPU:** NVIDIA GeForce RTX 4090  
**Working directory inside Docker:** `/app`

This notebook is designed to collect the measurements required for Parts A–F while writing raw results into the repository folders:

- `/app/results/gpu_info/`
- `/app/results/matmul/`
- `/app/results/bandwidth/`
- `/app/results/attention/`
- `/app/results/thermal/`
- `/app/figures/`

> Important: Do not modify system settings or install software on the lab workstation.  
> This notebook assumes the provided PyTorch Docker image is already running with `--gpus all`.

Run cells **in order**. Do not start Part E until Parts A–D have completed and results have been checked.

## Configuration

The assignment requires achieved TFLOPS as a percentage of the card's theoretical peak.

Fill in the theoretical peak values you are using **from the vendor source you cite in `METRICS.md`**.

Do not mix sparse and dense peak values. Use the same convention consistently and state it in your report.

In [ ]:
# Edit these using the theoretical values from the vendor documentation/source
# you choose to cite in METRICS.md.
#
# Leave as None until you have verified the values.
THEORETICAL_TFLOPS = {
    "FP32": None,
    "TF32": None,
    "FP16": None,
    "BF16": None,
}

MATRIX_SIZES = [1024, 4096, 8192, 16384]
WARMUP_REPS = 10
TIMED_REPS = 30

# Attention experiment configuration.
ATTN_DTYPE = torch.float16
ATTN_BATCH = 8
ATTN_HEADS = 1
ATTN_HEAD_DIM = 64
ATTN_SEQ_LENGTHS = [512, 1024, 2048, 4096, 8192, 16384]

print("Configuration loaded.")

# Part A — Onboarding and provenance

This section records the GPU UUID, driver, VRAM, power limit, PyTorch/CUDA versions, and complete `nvidia-smi -q` output.

In [ ]:
assert torch.cuda.is_available(), "CUDA is not available. Stop and fix the container/GPU setup."

gpu_name = torch.cuda.get_device_name(0)

query_cmd = [
    "nvidia-smi",
    "--query-gpu=name,uuid,driver_version,memory.total,power.limit",
    "--format=csv,noheader,nounits",
]
raw = subprocess.check_output(query_cmd, text=True).strip()
parts = [x.strip() for x in raw.split(",")]

gpu_record = {
    "name": parts[0],
    "uuid": parts[1],
    "driver_version": parts[2],
    "memory_total_mib": parts[3],
    "power_limit_w": parts[4],
    "pytorch_version": torch.__version__,
    "pytorch_cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
}

gpu_record

In [ ]:
# Save complete nvidia-smi -q output.
with open(GPU_INFO_DIR / "nvidia_smi_q.txt", "w", encoding="utf-8") as f:
    subprocess.run(["nvidia-smi", "-q"], stdout=f, text=True, check=True)

# Save compact software/hardware record.
with open(GPU_INFO_DIR / "software_stack.txt", "w", encoding="utf-8") as f:
    for k, v in gpu_record.items():
        f.write(f"{k}: {v}\n")

print((GPU_INFO_DIR / "software_stack.txt").read_text())
print("Saved:", GPU_INFO_DIR / "nvidia_smi_q.txt")

### Vendor documentation fields to record manually in `METRICS.md`

Record and cite:

- Architecture
- Memory type
- Specified memory bandwidth
- Tensor Core generation
- Reduced precisions supported by Tensor Cores
- Theoretical peak used for FP32 / TF32 / FP16 / BF16 comparisons

This notebook intentionally does not invent or hard-code those documentation values.

# Part B — Precision and achieved throughput

For an `N × N` dense matrix multiplication, the approximate work is:

\[
2N^3 \text{ FLOPs}
\]

Achieved throughput:

\[
\text{TFLOPS} = \frac{2N^3}{t \times 10^{12}}
\]

CUDA events are used for GPU timing.

In [ ]:
def _make_matmul_inputs(n, dtype, device="cuda"):
    a = torch.randn((n, n), device=device, dtype=dtype)
    b = torch.randn((n, n), device=device, dtype=dtype)
    return a, b


def benchmark_matmul_one(n, precision, warmups=WARMUP_REPS, reps=TIMED_REPS):
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    if precision == "FP32":
        dtype = torch.float32
        torch.set_float32_matmul_precision("highest")
    elif precision == "TF32":
        dtype = torch.float32
        torch.set_float32_matmul_precision("high")
    elif precision == "FP16":
        dtype = torch.float16
    elif precision == "BF16":
        dtype = torch.bfloat16
    else:
        raise ValueError(precision)

    a, b = _make_matmul_inputs(n, dtype)

    # Warmup
    for _ in range(warmups):
        _ = a @ b
    torch.cuda.synchronize()

    times_ms = []
    for _ in range(reps):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        c = a @ b
        end.record()

        torch.cuda.synchronize()
        times_ms.append(start.elapsed_time(end))

    mean_ms = float(np.mean(times_ms))
    std_ms = float(np.std(times_ms, ddof=1)) if len(times_ms) > 1 else 0.0
    seconds = mean_ms / 1000.0
    tflops = (2.0 * (n ** 3)) / seconds / 1e12

    theoretical = THEORETICAL_TFLOPS.get(precision)
    pct_theoretical = None if theoretical in (None, 0) else 100.0 * tflops / theoretical

    del a, b, c
    torch.cuda.empty_cache()

    return {
        "gpu_uuid": gpu_record["uuid"],
        "gpu_name": gpu_record["name"],
        "precision": precision,
        "N": n,
        "warmup_reps": warmups,
        "timed_reps": reps,
        "mean_ms": mean_ms,
        "std_ms": std_ms,
        "achieved_tflops": tflops,
        "theoretical_tflops": theoretical,
        "percent_theoretical": pct_theoretical,
    }

In [ ]:
matmul_rows = []

for precision in ["FP32", "TF32", "FP16", "BF16"]:
    for n in MATRIX_SIZES:
        print(f"Running {precision}, N={n} ...", flush=True)
        try:
            row = benchmark_matmul_one(n, precision)
            matmul_rows.append(row)
            print(
                f"  mean={row['mean_ms']:.3f} ms, "
                f"TFLOPS={row['achieved_tflops']:.2f}"
            )
        except torch.OutOfMemoryError as e:
            torch.cuda.empty_cache()
            matmul_rows.append({
                "gpu_uuid": gpu_record["uuid"],
                "gpu_name": gpu_record["name"],
                "precision": precision,
                "N": n,
                "warmup_reps": WARMUP_REPS,
                "timed_reps": TIMED_REPS,
                "mean_ms": np.nan,
                "std_ms": np.nan,
                "achieved_tflops": np.nan,
                "theoretical_tflops": THEORETICAL_TFLOPS.get(precision),
                "percent_theoretical": np.nan,
                "status": "OOM",
            })
            print("  OOM")

matmul_df = pd.DataFrame(matmul_rows)
matmul_df.to_csv(MATMUL_DIR / "matmul_results.csv", index=False)
matmul_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for precision, group in matmul_df.groupby("precision"):
    ok = group.dropna(subset=["achieved_tflops"]).sort_values("N")
    ax.plot(ok["N"], ok["achieved_tflops"], marker="o", label=precision)

ax.set_xlabel("Matrix size N")
ax.set_ylabel("Achieved TFLOPS")
ax.set_title("Dense Matmul Throughput vs Matrix Size")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "matmul_tflops.png", dpi=160)
plt.show()

### Plateau interpretation

Use the plotted results to identify the first matrix size where each precision is effectively near its maximum measured throughput.

Small matrices generally do not reach peak throughput because launch/scheduling overhead and insufficient parallel work prevent the GPU from being fully occupied.

## Optional lower-precision experiment

If your stack exposes FP8 or another lower precision, test it here.

If the installed PyTorch/CUDA stack does not expose a usable GEMM path, record exactly what you tried and the exception/error message. That is a valid finding for this assignment.

In [ ]:
# This cell only inspects what the installed PyTorch exposes.
# It does not assume that an exposed dtype has a supported GEMM kernel.

lower_precision_attempt = {
    "torch_float8_e4m3fn_present": hasattr(torch, "float8_e4m3fn"),
    "torch_float8_e5m2_present": hasattr(torch, "float8_e5m2"),
}

lower_precision_attempt

# Part C — Bandwidth-bound vs compute-bound

We measure:

1. A large elementwise add as a memory-bound operation.
2. A large matmul as a compute-bound operation.

For `C = A + B` in FP32:

- read A: 4 bytes
- read B: 4 bytes
- write C: 4 bytes
- approximately 1 FLOP

Approximate arithmetic intensity:

\[
1 / 12 \approx 0.0833 \text{ FLOP/byte}
\]

In [ ]:
def benchmark_elementwise_add(num_elements=128_000_000, dtype=torch.float32, warmups=10, reps=30):
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    a = torch.randn(num_elements, device="cuda", dtype=dtype)
    b = torch.randn(num_elements, device="cuda", dtype=dtype)

    for _ in range(warmups):
        c = a + b
    torch.cuda.synchronize()

    times_ms = []
    for _ in range(reps):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        c = a + b
        end.record()
        torch.cuda.synchronize()

        times_ms.append(start.elapsed_time(end))

    mean_ms = float(np.mean(times_ms))
    sec = mean_ms / 1000.0

    bytes_per_element = torch.tensor([], dtype=dtype).element_size()
    bytes_moved = num_elements * bytes_per_element * 3  # read A + read B + write C
    effective_gbps = bytes_moved / sec / 1e9
    arithmetic_intensity = num_elements / bytes_moved

    del a, b, c
    torch.cuda.empty_cache()

    return {
        "gpu_uuid": gpu_record["uuid"],
        "operation": "elementwise_add",
        "dtype": str(dtype),
        "num_elements": num_elements,
        "warmup_reps": warmups,
        "timed_reps": reps,
        "mean_ms": mean_ms,
        "bytes_moved": bytes_moved,
        "effective_bandwidth_gbs": effective_gbps,
        "arithmetic_intensity_flops_per_byte": arithmetic_intensity,
    }


bw_row = benchmark_elementwise_add()
bw_df = pd.DataFrame([bw_row])
bw_df.to_csv(BANDWIDTH_DIR / "bandwidth_results.csv", index=False)
bw_df

In [ ]:
# Compute arithmetic intensity for a representative square FP32 matmul.
# Approximation:
# FLOPs = 2N^3
# minimum bytes for A, B, C = 3 * N^2 * bytes_per_element
# This simple estimate ignores cache hierarchy details and repeated traffic.

N_ROOFLINE = 8192
fp32_bytes = 4

matmul_ai = (2 * N_ROOFLINE**3) / (3 * N_ROOFLINE**2 * fp32_bytes)

roofline_df = pd.DataFrame([
    {
        "gpu_uuid": gpu_record["uuid"],
        "operation": "elementwise_add_fp32",
        "arithmetic_intensity_flops_per_byte": bw_row["arithmetic_intensity_flops_per_byte"],
        "expected_side": "memory-bound",
    },
    {
        "gpu_uuid": gpu_record["uuid"],
        "operation": f"matmul_fp32_N{N_ROOFLINE}",
        "arithmetic_intensity_flops_per_byte": matmul_ai,
        "expected_side": "compute-bound",
    },
])

roofline_df.to_csv(BANDWIDTH_DIR / "arithmetic_intensity.csv", index=False)
roofline_df

To compute **effective bandwidth as a percentage of specified bandwidth**, enter the vendor-specified memory bandwidth below.

In [ ]:
SPECIFIED_MEMORY_BANDWIDTH_GBS = None  # fill from cited vendor documentation

if SPECIFIED_MEMORY_BANDWIDTH_GBS:
    bw_df["percent_specified_bandwidth"] = (
        100.0 * bw_df["effective_bandwidth_gbs"] / SPECIFIED_MEMORY_BANDWIDTH_GBS
    )
else:
    bw_df["percent_specified_bandwidth"] = np.nan

bw_df.to_csv(BANDWIDTH_DIR / "bandwidth_results.csv", index=False)
bw_df

# Part D — The cost of attention

Naive scaled dot-product attention explicitly materializes the full `L × L` score/probability matrix.

Configuration used here:

- dtype: FP16
- batch size: 8
- heads: 1
- head dimension: 64

The assignment's coarse sweep is:

`512, 1024, 2048, 4096, 8192, 16384`

In [ ]:
def make_qkv(seq_len, dtype=ATTN_DTYPE):
    shape = (ATTN_BATCH, ATTN_HEADS, seq_len, ATTN_HEAD_DIM)
    q = torch.randn(shape, device="cuda", dtype=dtype)
    k = torch.randn(shape, device="cuda", dtype=dtype)
    v = torch.randn(shape, device="cuda", dtype=dtype)
    return q, k, v


def naive_attention(q, k, v):
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(q.size(-1))
    probs = torch.softmax(scores, dim=-1)
    out = torch.matmul(probs, v)
    return out


def fused_attention(q, k, v):
    return F.scaled_dot_product_attention(q, k, v)


def benchmark_attention_one(seq_len, implementation="naive", warmups=3, reps=10):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    q, k, v = make_qkv(seq_len)

    fn = naive_attention if implementation == "naive" else fused_attention

    try:
        for _ in range(warmups):
            out = fn(q, k, v)
        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()

        times_ms = []
        for _ in range(reps):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            out = fn(q, k, v)
            end.record()
            torch.cuda.synchronize()

            times_ms.append(start.elapsed_time(end))

        peak_bytes = torch.cuda.max_memory_allocated()
        mean_ms = float(np.mean(times_ms))

        result = {
            "gpu_uuid": gpu_record["uuid"],
            "implementation": implementation,
            "dtype": str(ATTN_DTYPE),
            "batch": ATTN_BATCH,
            "heads": ATTN_HEADS,
            "head_dim": ATTN_HEAD_DIM,
            "sequence_length": seq_len,
            "warmup_reps": warmups,
            "timed_reps": reps,
            "mean_ms": mean_ms,
            "std_ms": float(np.std(times_ms, ddof=1)) if len(times_ms) > 1 else 0.0,
            "peak_memory_bytes": peak_bytes,
            "peak_memory_gib": peak_bytes / (1024**3),
            "status": "SUCCESS",
        }

        del q, k, v, out
        torch.cuda.empty_cache()
        return result

    except torch.OutOfMemoryError:
        try:
            del q, k, v
        except Exception:
            pass
        torch.cuda.empty_cache()

        return {
            "gpu_uuid": gpu_record["uuid"],
            "implementation": implementation,
            "dtype": str(ATTN_DTYPE),
            "batch": ATTN_BATCH,
            "heads": ATTN_HEADS,
            "head_dim": ATTN_HEAD_DIM,
            "sequence_length": seq_len,
            "warmup_reps": warmups,
            "timed_reps": reps,
            "mean_ms": np.nan,
            "std_ms": np.nan,
            "peak_memory_bytes": np.nan,
            "peak_memory_gib": np.nan,
            "status": "OOM",
        }

In [ ]:
naive_rows = []

for L in ATTN_SEQ_LENGTHS:
    print(f"Naive attention L={L} ...", flush=True)
    row = benchmark_attention_one(L, implementation="naive")
    naive_rows.append(row)
    print(row["status"], row.get("mean_ms"), row.get("peak_memory_gib"))

naive_df = pd.DataFrame(naive_rows)
naive_df.to_csv(ATTN_DIR / "attention_naive_results.csv", index=False)
naive_df

## Refine the naive OOM boundary

The assignment asks for:

- largest tested sequence length that succeeds
- smallest tested sequence length that fails

Do not claim an exact single-token boundary unless you actually search at single-token resolution.

The helper below performs a configurable interval refinement using a chosen step.

In [ ]:
def refine_oom_boundary(implementation, low_success, high_failure, step=256):
    assert low_success < high_failure
    tested = []

    for L in range(low_success + step, high_failure, step):
        print(f"{implementation}: testing L={L}", flush=True)
        row = benchmark_attention_one(L, implementation=implementation, warmups=1, reps=3)
        tested.append(row)
        print(" ", row["status"])

        if row["status"] == "OOM":
            break

    successes = [r["sequence_length"] for r in tested if r["status"] == "SUCCESS"]
    failures = [r["sequence_length"] for r in tested if r["status"] == "OOM"]

    largest_success = max([low_success] + successes) if successes else low_success
    smallest_failure = min([high_failure] + failures) if failures else high_failure

    return pd.DataFrame(tested), largest_success, smallest_failure


# Example only:
# refined_naive_df, naive_largest_success, naive_smallest_failure = #     refine_oom_boundary("naive", low_success=8192, high_failure=16384, step=256)
#
# refined_naive_df.to_csv(ATTN_DIR / "attention_naive_oom_refinement.csv", index=False)
#
# Change low_success/high_failure to match YOUR coarse sweep before running.

## Fit peak-memory growth for naive attention

The fit uses:

\[
M(L) = aL^2 + bL + c
\]

and reports the measured quadratic coefficient `a`.

In [ ]:
naive_success = naive_df[
    (naive_df["status"] == "SUCCESS") &
    naive_df["peak_memory_gib"].notna()
].copy()

if len(naive_success) >= 3:
    coeff = np.polyfit(
        naive_success["sequence_length"].to_numpy(dtype=float),
        naive_success["peak_memory_gib"].to_numpy(dtype=float),
        2
    )
    a, b, c = coeff
    print(f"Quadratic fit: M(L) = {a:.12e} L^2 + {b:.12e} L + {c:.12e}")
    print(f"Measured quadratic coefficient a = {a:.12e} GiB/token^2")

    x = np.linspace(
        naive_success["sequence_length"].min(),
        naive_success["sequence_length"].max(),
        300
    )
    y = a*x*x + b*x + c

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(naive_success["sequence_length"], naive_success["peak_memory_gib"], label="Measured")
    ax.plot(x, y, label="Quadratic fit")
    ax.set_xlabel("Sequence length")
    ax.set_ylabel("Peak allocated memory (GiB)")
    ax.set_title("Naive Attention Peak Memory")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / "attention_memory.png", dpi=160)
    plt.show()
else:
    print("Need at least 3 successful measurements for a quadratic fit.")

## Fused / memory-efficient attention

PyTorch's `scaled_dot_product_attention` can dispatch to optimized CUDA implementations depending on dtype, shape, PyTorch version, and GPU support.

This section measures the same sequence lengths using `torch.nn.functional.scaled_dot_product_attention`.

In [ ]:
fused_rows = []

for L in ATTN_SEQ_LENGTHS:
    print(f"Fused attention L={L} ...", flush=True)
    row = benchmark_attention_one(L, implementation="fused")
    fused_rows.append(row)
    print(row["status"], row.get("mean_ms"), row.get("peak_memory_gib"))

fused_df = pd.DataFrame(fused_rows)
fused_df.to_csv(ATTN_DIR / "attention_fused_results.csv", index=False)
fused_df

In [ ]:
comparison = naive_df.merge(
    fused_df,
    on=["gpu_uuid", "dtype", "batch", "heads", "head_dim", "sequence_length"],
    suffixes=("_naive", "_fused"),
)

comparison["speedup_naive_over_fused"] = (
    comparison["mean_ms_naive"] / comparison["mean_ms_fused"]
)

comparison.to_csv(ATTN_DIR / "attention_comparison.csv", index=False)
comparison[
    [
        "sequence_length",
        "status_naive",
        "status_fused",
        "mean_ms_naive",
        "mean_ms_fused",
        "speedup_naive_over_fused",
        "peak_memory_gib_naive",
        "peak_memory_gib_fused",
    ]
]

### Fused-attention explanation for your report

A fused or memory-efficient attention kernel avoids materializing the complete sequence-by-sequence attention matrix in global GPU memory. Instead, it computes attention in blocks, combines the score, softmax, and value-accumulation work more tightly, and reduces intermediate reads/writes. This lowers memory traffic and peak intermediate storage. The result is usually a larger feasible sequence length and often lower forward latency.

# Part E — Sustained load and thermal behavior

**Do not run this until you are ready for an uninterrupted ~20-minute experiment.**

The experiment:

- samples every 5 seconds
- records GPU clock, memory clock, temperature, power draw, and utilization
- runs a sustained matmul load
- records throughput observations
- writes `/app/results/thermal/thermal_log.csv`

This is intentionally a long-running cell.

In [ ]:
def query_gpu_sample():
    cmd = [
        "nvidia-smi",
        "--query-gpu=timestamp,uuid,clocks.gr,clocks.mem,temperature.gpu,power.draw,utilization.gpu",
        "--format=csv,noheader,nounits",
    ]
    line = subprocess.check_output(cmd, text=True).strip()
    fields = [x.strip() for x in line.split(",")]
    return {
        "timestamp": fields[0],
        "gpu_uuid": fields[1],
        "graphics_clock_mhz": float(fields[2]),
        "memory_clock_mhz": float(fields[3]),
        "temperature_c": float(fields[4]),
        "power_w": float(fields[5]),
        "utilization_percent": float(fields[6]),
    }


def run_thermal_experiment(duration_s=1200, sample_interval_s=5, n=8192, dtype=torch.float16):
    torch.cuda.empty_cache()
    a = torch.randn((n, n), device="cuda", dtype=dtype)
    b = torch.randn((n, n), device="cuda", dtype=dtype)

    # Warmup
    for _ in range(10):
        _ = a @ b
    torch.cuda.synchronize()

    telemetry = []
    throughput_samples = []

    stop_event = threading.Event()

    def monitor():
        t0 = time.perf_counter()
        while not stop_event.is_set():
            sample = query_gpu_sample()
            sample["elapsed_s"] = time.perf_counter() - t0
            telemetry.append(sample)
            stop_event.wait(sample_interval_s)

    monitor_thread = threading.Thread(target=monitor, daemon=True)
    monitor_thread.start()

    t0 = time.perf_counter()

    try:
        while (time.perf_counter() - t0) < duration_s:
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            c = a @ b
            end.record()
            torch.cuda.synchronize()

            ms = start.elapsed_time(end)
            sec = ms / 1000.0
            tflops = (2.0 * n**3) / sec / 1e12

            throughput_samples.append({
                "elapsed_s": time.perf_counter() - t0,
                "matmul_ms": ms,
                "tflops": tflops,
            })
    finally:
        stop_event.set()
        monitor_thread.join(timeout=10)
        del a, b, c
        torch.cuda.empty_cache()

    telemetry_df = pd.DataFrame(telemetry)
    throughput_df = pd.DataFrame(throughput_samples)

    telemetry_df.to_csv(THERMAL_DIR / "thermal_log.csv", index=False)
    throughput_df.to_csv(THERMAL_DIR / "thermal_throughput.csv", index=False)

    return telemetry_df, throughput_df


# RUN ONLY WHEN READY:
# thermal_df, thermal_tp_df = run_thermal_experiment(
#     duration_s=1200,
#     sample_interval_s=5,
#     n=8192,
#     dtype=torch.float16,
# )

After the thermal experiment finishes, run the next cells.

In [ ]:
# Run after thermal_df exists.

if "thermal_df" in globals():
    fig, ax1 = plt.subplots(figsize=(9, 5))

    ax1.plot(thermal_df["elapsed_s"], thermal_df["graphics_clock_mhz"], label="GPU clock (MHz)")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("GPU clock (MHz)")
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(thermal_df["elapsed_s"], thermal_df["temperature_c"], label="Temperature (C)")
    ax2.set_ylabel("Temperature (°C)")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")

    ax1.set_title("Sustained Load: Clock and Temperature vs Time")
    fig.tight_layout()
    fig.savefig(FIGURES / "thermal_behavior.png", dpi=160)
    plt.show()
else:
    print("Run the thermal experiment first.")

In [ ]:
# Peak throughput in first 30 seconds vs steady-state throughput in final 5 minutes.

if "thermal_tp_df" in globals():
    first_30 = thermal_tp_df[thermal_tp_df["elapsed_s"] <= 30]
    final_5min = thermal_tp_df[thermal_tp_df["elapsed_s"] >= 900]

    peak_first_30 = first_30["tflops"].max()
    steady_final_5min = final_5min["tflops"].mean()
    steady_as_pct_peak = 100.0 * steady_final_5min / peak_first_30

    thermal_summary = pd.DataFrame([{
        "gpu_uuid": gpu_record["uuid"],
        "peak_tflops_first_30s": peak_first_30,
        "steady_tflops_final_5min_mean": steady_final_5min,
        "steady_state_as_percent_peak": steady_as_pct_peak,
    }])

    thermal_summary.to_csv(THERMAL_DIR / "thermal_summary.csv", index=False)
    thermal_summary
else:
    print("Run the thermal experiment first.")

### Throttling interpretation

Use the telemetry plot and `nvidia-smi -q` limits to identify whether:

- graphics clock drops materially during sustained load,
- temperature reaches a stable ceiling,
- power draw approaches the reported power limit,
- utilization remains high.

Report the observed onset time and whether the behavior is best associated with a thermal ceiling, power ceiling, or neither.

Do not claim throttling solely because the temperature increased.

# Part F — Summary table

This helper builds as much of the summary table as possible from saved measurements.

Some fields, especially refined OOM boundaries and throttle onset, should be filled after reviewing your experimental results.

In [ ]:
summary = {
    "GPU UUID": gpu_record["uuid"],
    "GPU": gpu_record["name"],
    "Peak achieved TFLOPS (BF16)": None,
    "% of theoretical peak (BF16)": None,
    "Effective bandwidth (GB/s)": None,
    "Naive attention OOM length": None,
    "Fused attention OOM length": None,
    "Steady-state / peak throughput (%)": None,
    "Throttle onset (s, or none)": None,
}

# Populate what is available.
if "matmul_df" in globals():
    bf16_ok = matmul_df[
        (matmul_df["precision"] == "BF16") &
        matmul_df["achieved_tflops"].notna()
    ]
    if len(bf16_ok):
        idx = bf16_ok["achieved_tflops"].idxmax()
        summary["Peak achieved TFLOPS (BF16)"] = float(matmul_df.loc[idx, "achieved_tflops"])
        if pd.notna(matmul_df.loc[idx, "percent_theoretical"]):
            summary["% of theoretical peak (BF16)"] = float(matmul_df.loc[idx, "percent_theoretical"])

if "bw_df" in globals() and len(bw_df):
    summary["Effective bandwidth (GB/s)"] = float(bw_df.iloc[0]["effective_bandwidth_gbs"])

if "thermal_summary" in globals():
    summary["Steady-state / peak throughput (%)"] = float(
        thermal_summary.iloc[0]["steady_state_as_percent_peak"]
    )

summary_df = pd.DataFrame(
    [{"Measurement": k, "Your GPU": v, "Notes": ""} for k, v in summary.items()]
)

summary_df.to_csv(RESULTS / "summary_table.csv", index=False)
summary_df

# Run log helper

Use this helper after each major section to append a traceable note to `/app/RUN_LOG.txt`.

In [ ]:
from datetime import datetime

def append_run_log(section, notes):
    path = ROOT / "RUN_LOG.txt"
    timestamp = datetime.now().isoformat(timespec="seconds")

    with open(path, "a", encoding="utf-8") as f:
        f.write("=" * 70 + "\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"Section: {section}\n")
        f.write(f"GPU UUID: {gpu_record['uuid']}\n")
        f.write(f"GPU: {gpu_record['name']}\n")
        f.write(f"PyTorch: {torch.__version__}\n")
        f.write(f"CUDA: {torch.version.cuda}\n")
        f.write(f"Notes: {notes}\n")

    print("Appended to", path)


# Examples:
# append_run_log("Part A", "Captured nvidia-smi -q and software stack.")
# append_run_log("Part B", "Completed FP32/TF32/FP16/BF16 matmul sweep.")

# Final submission checklist

Before leaving the GPU lab, verify that these exist:

### Required provenance / logs
- `results/gpu_info/nvidia_smi_q.txt`
- `results/gpu_info/software_stack.txt`
- `RUN_LOG.txt`
- `reservation/GPU_HOURS.md`

### Part B
- `results/matmul/matmul_results.csv`
- `figures/matmul_tflops.png`

### Part C
- `results/bandwidth/bandwidth_results.csv`
- `results/bandwidth/arithmetic_intensity.csv`

### Part D
- `results/attention/attention_naive_results.csv`
- `results/attention/attention_fused_results.csv`
- OOM refinement CSV(s)
- `results/attention/attention_comparison.csv`
- `figures/attention_memory.png`

### Part E
- `results/thermal/thermal_log.csv`
- `results/thermal/thermal_throughput.csv`
- `results/thermal/thermal_summary.csv`
- `figures/thermal_behavior.png`

### Part F
- `results/summary_table.csv`
- `METRICS.md`
- `AI_USE.md`

Before deleting the lab workstation copy, back up all results using an allowed method such as GitHub browser upload or another cloud option approved by the lab.